In [27]:
import os
import json
import glob
from shapely.geometry import shape, mapping
from shapely.affinity import scale
from pathlib import Path

# --- CONFIGURACIÓN ---
RAW_DATA_DIR = os.path.join("C:/Users/paez/Documents/Doct_2026/ratitas_2026")
WSI_IMAGES_DIR = os.path.join(RAW_DATA_DIR, "data/raw/wsi_images")
ANNOTATIONS_DIR = os.path.join(RAW_DATA_DIR, "data/raw/annotations")
OUTPUT_DIR = os.path.join(RAW_DATA_DIR,"data/raw/annotations_resized")
METADATA_DIR = os.path.join(RAW_DATA_DIR,"data/raw/metadata")
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
def scale_geojson_files():
    image_files = sorted(glob.glob(os.path.join(WSI_IMAGES_DIR, "*.tif")))
    geojson_files = sorted(glob.glob(os.path.join(ANNOTATIONS_DIR, "*.geojson")))
    metajson_files = sorted(glob.glob(os.path.join(METADATA_DIR, "*.meta.json")))

    for img_p, geo_p, meta_p in zip(image_files, geojson_files, metajson_files):
        print(f"Procesando: {os.path.basename(img_p)}...")

        # 1. Obtener dimensiones reales del .tif
        with tiffslide.TiffSlide(img_p) as slide:
            current_width = slide.dimensions[0]
            current_height = slide.dimensions[1]
            # Leemos la imagen completa para visualizarla (Cuidado si es muy grande, 8k está bien para RAM)
            img_data = slide.read_region((0, 0), 0, slide.dimensions).convert("RGB")

        # 2. Obtener el ancho original del metadato (Nivel 0)
        with open(meta_p, 'r') as f:
            data_meta = json.load(f)
        original_width = data_meta['width_px']

        # 3. Calcular factor de escala
        DOWNSAMPLE_FACTOR = current_width / original_width
        print(f"  - Factor calculado: {DOWNSAMPLE_FACTOR:.6f}")

        # 4. Escalar las anotaciones
        with open(geo_p, 'r') as f:
            data_geojson = json.load(f)

        #fig, ax = plt.subplots(figsize=(12, 8))
        #ax.imshow(img_data)
        #ax.set_title(f"Verificación: {os.path.basename(img_p)}")

        if 'features' in data_geojson:
            for feature in data_geojson['features']:
                geom = shape(feature['geometry'])
                
                # ESCALADO
                scaled_geom = scale(geom, xfact=DOWNSAMPLE_FACTOR, yfact=DOWNSAMPLE_FACTOR, origin=(0,0))
                
                # Visualización: Dibujar el polígono escalado
                #if scaled_geom.geom_type == 'Polygon':
                #    x, y = scaled_geom.exterior.xy
                #    ax.plot(x, y, color='red', linewidth=1)
                #elif scaled_geom.geom_type == 'MultiPolygon':
                #    for poly in scaled_geom.geoms:
                #        x, y = poly.exterior.xy
                #        ax.plot(x, y, color='red', linewidth=1)
                
                # Reemplazar en el dict (opcional si solo visualizas, pero lo mantenemos por consistencia)
                feature['geometry'] = mapping(scaled_geom)

        #plt.axis('off')
        #plt.show()

        # 5. Guardar el resultado corregido (COMENTADO POR SEGURIDAD)
        
        #output_path = os.path.join(OUTPUT_DIR, os.path.basename(geo_p))
        #with open(output_path, 'w') as f:
        #    json.dump(data_geojson, f, indent=2)
        #print(f"  - Guardado en: {output_path}")
        

    print(f"\n✅ Visualización terminada.")

# Ejecutar
scale_geojson_files()

Procesando: slide_001.tif...
  - Factor calculado: 0.249993
  - Guardado en: C:/Users/paez/Documents/Doct_2026/ratitas_2026\data/raw/annotations_resized\slide_001.geojson
Procesando: slide_002.tif...
  - Factor calculado: 0.249994
  - Guardado en: C:/Users/paez/Documents/Doct_2026/ratitas_2026\data/raw/annotations_resized\slide_002.geojson
Procesando: slide_003.tif...
  - Factor calculado: 0.250000
  - Guardado en: C:/Users/paez/Documents/Doct_2026/ratitas_2026\data/raw/annotations_resized\slide_003.geojson
Procesando: slide_004.tif...
  - Factor calculado: 0.250014
  - Guardado en: C:/Users/paez/Documents/Doct_2026/ratitas_2026\data/raw/annotations_resized\slide_004.geojson
Procesando: slide_005.tif...
  - Factor calculado: 0.250007
  - Guardado en: C:/Users/paez/Documents/Doct_2026/ratitas_2026\data/raw/annotations_resized\slide_005.geojson
Procesando: slide_006.tif...
  - Factor calculado: 0.250000
  - Guardado en: C:/Users/paez/Documents/Doct_2026/ratitas_2026\data/raw/annotations_

In [9]:
import glob, os
geojson_files = sorted(glob.glob(os.path.join("C:/Users/paez/Documents/Doct_2026/ratitas_2026/data/raw/annotations_resized", "*.geojson")))

In [11]:
# Visualización de anotaciones resizeadas

for g in geojson_files:
    clases_detectadas = set()
    import json

    with open(g, 'r') as f: # geojson_p es la ruta a uno de tus archivos resized
        data = json.load(f)
        for feature in data.get('features', []):
            # Buscamos el nombre de la clasificación
            prop = feature.get('properties', {})
            class_name = prop.get('name', 'Sin Clase')
            clases_detectadas.add(class_name)

    print(f"Tus clases registradas son: {clases_detectadas}")
    print('#'*20)

Tus clases registradas son: {'Labyrinth', 'Decidua', 'Junctional Zone'}
####################
Tus clases registradas son: {'Labyrinth', 'Decidua', 'Junctional Zone'}
####################
Tus clases registradas son: {'Labyrinth', 'Decidua', 'Junctional Zone'}
####################
Tus clases registradas son: {'Labyrinth', 'Decidua', 'Junctional Zone'}
####################
Tus clases registradas son: {'Labyrinth', 'Decidua', 'Junctional Zone'}
####################
Tus clases registradas son: {'Labyrinth', 'Decidua', 'Junctional Zone'}
####################
Tus clases registradas son: {'Sin Clase', 'Decidua', 'Junctional Zone'}
####################
Tus clases registradas son: {'Labyrinth', 'Decidua', 'Junctional Zone'}
####################
Tus clases registradas son: {'Labyrinth', 'Decidua', 'Junctional Zone'}
####################
Tus clases registradas son: {'Labyrinth', 'Decidua', 'Junctional Zone'}
####################
Tus clases registradas son: {'Labyrinth', 'Decidua', 'Junctional Zone'